# 02 - Mercado laboral (EPH)

Indicadores del mercado de trabajo a partir de la EPH (INDEC), usando los parquets
compilados por el notebook **00_preparacion_bases**.

**Contenido:**
1. Tasas básicas (actividad, empleo, desocupación) — serie completa 2017-2025.
2. Subocupación horaria (`INTENSI`) — serie completa.
3. Categoría ocupacional de los ocupados (`CAT_OCUP`).
4. Informalidad laboral (`EMPLEO`/`SECTOR`) — **solo desde 2023T4** (quiebre de esquema).

**Definiciones (INDEC), todo ponderado con `PONDERA`:**
- Población económicamente activa (PEA) = ocupados + desocupados (`ESTADO` ∈ {1, 2}).
- **Tasa de actividad** = PEA / población total.
- **Tasa de empleo** = ocupados / población total.
- **Tasa de desocupación** = desocupados / PEA.
- **Tasa de subocupación** = subocupados (`INTENSI`==1) / PEA.

> `ESTADO`: 1=Ocupado, 2=Desocupado, 3=Inactivo, 4=Menor de 10. Ver `.claude/memoria_EPH.md`.

## 1. Setup (Colab)

Clona el repo, monta Drive y copia los parquets compilados a disco local (evita la
desconexión del mount al leer muchos archivos). Requiere haber corrido el notebook 00.

In [ ]:
import sys, os, shutil, glob

REPO_URL = "https://github.com/santiagoriverti/analisis_EPH.git"
REPO_DIR = "/content/analisis_EPH"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

from google.colab import drive
drive.mount('/content/drive')

# Copiar parquets de Drive a disco local (estable y rápido).
DRIVE_PROCESSED = "/content/drive/MyDrive/carga_EPH/processed"
PROCESSED_DIR = "/content/processed_local"
os.makedirs(PROCESSED_DIR, exist_ok=True)
for src in glob.glob(os.path.join(DRIVE_PROCESSED, "eph_T*.parquet")):
    dst = os.path.join(PROCESSED_DIR, os.path.basename(src))
    if not os.path.exists(dst):
        shutil.copy(src, dst)
n = len(glob.glob(os.path.join(PROCESSED_DIR, "eph_T*.parquet")))
print(f"Parquets locales listos en {PROCESSED_DIR}: {n} trimestres")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_panel, list_available_quarters, _period_tag

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

quarters = list_available_quarters()
ULTIMO = quarters[-1]
print("Último trimestre:", _period_tag(*ULTIMO))

## 2. Tasas básicas (serie 2017-2025)

Actividad, empleo y desocupación por trimestre, ponderadas con `PONDERA`.

In [ ]:
lab = load_panel(columns=["ESTADO", "INTENSI", "PONDERA"], out_dir=PROCESSED_DIR)

def tasas(g):
    w = g["PONDERA"]
    total = w.sum()
    ocupados = w[g["ESTADO"] == 1].sum()
    desocup = w[g["ESTADO"] == 2].sum()
    pea = ocupados + desocup
    suboc = w[g["INTENSI"] == 1].sum()
    return pd.Series({
        "actividad": pea / total * 100,
        "empleo": ocupados / total * 100,
        "desocupacion": desocup / pea * 100,
        "subocupacion": suboc / pea * 100,
    })

serie = lab.groupby(["ANIO", "TRIMESTRE"]).apply(tasas, include_groups=False)
serie.index = [f"{y}T{p}" for y, p in serie.index]
serie.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
for col, color in [("actividad", "#4C72B0"), ("empleo", "#55A868"), ("desocupacion", "#C44E52")]:
    serie[col].plot(ax=ax, marker="o", ms=4, label=col.capitalize(), color=color)
ax.set_ylabel("%")
ax.set_title("Tasas del mercado laboral (EPH, ponderado)")
ax.set_xticks(range(len(serie.index)))
ax.set_xticklabels(serie.index, rotation=90)
ax.legend()
plt.tight_layout()
plt.show()

## 3. Subocupación horaria

Tasa de subocupación (ocupados que trabajan menos horas de las que querrían,
`INTENSI`==1) sobre la PEA.

In [ ]:
ax = serie["subocupacion"].plot(marker="o", ms=4, color="#8172B3")
ax.set_ylabel("% de la PEA")
ax.set_title("Tasa de subocupación horaria (EPH, ponderado)")
ax.set_xticks(range(len(serie.index)))
ax.set_xticklabels(serie.index, rotation=90)
plt.tight_layout()
plt.show()

## 4. Categoría ocupacional de los ocupados

Distribución de la categoría ocupacional (`CAT_OCUP`) entre los ocupados, último trimestre.

In [ ]:
CAT_OCUP = {1: "Patrón", 2: "Cuenta propia", 3: "Obrero/empleado",
            4: "Trab. familiar s/remun.", 9: "Ns/Nr"}

co = load_panel(columns=["ESTADO", "CAT_OCUP", "PONDERA"], quarters=[ULTIMO], out_dir=PROCESSED_DIR)
ocup = co[co["ESTADO"] == 1]
dist = ocup.groupby("CAT_OCUP")["PONDERA"].sum()
dist.index = dist.index.map(lambda c: CAT_OCUP.get(int(c), str(c)))
dist = (dist / dist.sum() * 100).sort_values(ascending=True)

ax = dist.plot(kind="barh", color="#4C72B0")
ax.set_xlabel("% de los ocupados")
ax.set_title(f"Categoría ocupacional - {_period_tag(*ULTIMO)}")
for i, v in enumerate(dist):
    ax.text(v + 0.4, i, f"{v:.1f}%", va="center")
plt.tight_layout()
plt.show()

## 5. Informalidad laboral (desde 2023T4)

Las variables `EMPLEO` (1=Formal, 2=Informal) y `SECTOR` (1=Formal, 2=Informal, 3=Hogares)
se incorporaron en **4T2023** (quiebre de esquema). Por eso esta sección se restringe a los
trimestres ≥ 2023T4.

In [ ]:
# Trimestres con esquema nuevo (>= 2023T4)
q_nuevos = [(y, p) for (y, p) in quarters if (y, p) >= (2023, 4)]
print("Trimestres con informalidad:", [_period_tag(y, p) for y, p in q_nuevos])

inf = load_panel(columns=["ESTADO", "EMPLEO", "PONDERA"], quarters=q_nuevos, out_dir=PROCESSED_DIR)

def tasa_informal(g):
    ocup = g[g["ESTADO"] == 1]
    informales = ocup.loc[ocup["EMPLEO"] == 2, "PONDERA"].sum()
    total_ocup = ocup["PONDERA"].sum()
    return informales / total_ocup * 100

tasa_inf = inf.groupby(["ANIO", "TRIMESTRE"]).apply(tasa_informal, include_groups=False)
tasa_inf.index = [f"{y}T{p}" for y, p in tasa_inf.index]
print("\nTasa de informalidad (% de ocupados):")
tasa_inf.round(1)

In [ ]:
ax = tasa_inf.plot(kind="bar", color="#C44E52")
ax.set_ylabel("% de los ocupados")
ax.set_title("Tasa de informalidad laboral (ocupados informales / total ocupados)")
for i, v in enumerate(tasa_inf):
    ax.text(i, v + 0.3, f"{v:.1f}%", ha="center")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
Notebook sobre los parquets compilados por `00_preparacion_bases`. Las tasas siguen las
definiciones del INDEC (denominadores: población total para actividad/empleo, PEA para
desocupación/subocupación). Informalidad disponible solo desde 2023T4.